# FaceProof 07 — multi-seed + bootstrap CIs (Day 7)

Rebuild splits for `seeds:[13,37,71]`, retrain probes, report mean ± spread, and add percentile
bootstrap CIs over images for the seed-13 cross-generator AUROC. We load the **saved** manifest
(its row order matches the cached features) and only re-split it per seed.

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/Ezed9/faceProof.git"
BRANCH   = "main"
!git clone -b $BRANCH $REPO_URL
%cd faceProof
!pip install -q scikit-learn joblib pyyaml
import sys, os
sys.path.append(os.getcwd())

In [ ]:
from google.colab import drive
from pathlib import Path
import numpy as np, pandas as pd
drive.mount('/content/drive')
ROOT          = Path('/content/drive/MyDrive/faceproof')
CROPS_ROOT    = ROOT / 'data' / 'crops'
FEATURES_ROOT = ROOT / 'models' / 'features'
PROBES_ROOT   = ROOT / 'models' / 'probes'
REPORTS_ROOT  = ROOT / 'reports'
MANIFEST      = ROOT / 'data' / 'manifest.csv'
RESULTS_CSV   = REPORTS_ROOT / 'results.csv'
(REPORTS_ROOT / 'figures').mkdir(parents=True, exist_ok=True)
PROBES_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
import csv
RESULT_FIELDS = ['condition','train_gen','test_gen','corruption','strength','seed','metric','value']
def log_result(**row):
    full = {k: row.get(k, '') for k in RESULT_FIELDS}
    new = not RESULTS_CSV.exists()
    with open(RESULTS_CSV, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=RESULT_FIELDS)
        if new: w.writeheader()
        w.writerow(full)

## 2. Load features + saved manifest (drop its split; we re-split per seed)

In [ ]:
import yaml
from src.data import make_splits
from src.probe import train_probe, predict_proba
from src.metrics import auroc, eer, bootstrap_ci
X={'c4_clip':np.load(FEATURES_ROOT/'clip_all.npy'),'c1_resnet':np.load(FEATURES_ROOT/'resnet_all.npy')}
base=pd.read_csv(MANIFEST).drop(columns=['split'], errors='ignore')  # row order == feature order
for k,v in X.items():
    assert len(v)==len(base), f'{k} rows ({len(v)}) != manifest ({len(base)}); re-run nb 02'
pcfg=yaml.safe_load(open('configs/model.yaml'))['probe']
seeds=yaml.safe_load(open('configs/experiment.yaml'))['seeds']; print('seeds:',seeds)

## 3. Per-seed in-dist + cross-generator metrics

In [ ]:
def groups_for(df):
    rt=(df['label']==0)&(df['split']=='test_indist')
    return {'stylegan_indist':(df['split']=='test_indist').values,
            'stable_diffusion':(rt|(df['generator']=='stable_diffusion')).values}
rows=[]
for seed in seeds:
    df=make_splits(base.copy(), train_generators=['stylegan'], test_unseen=['stable_diffusion'], seed=seed)
    y=df['label'].values; m_tr=(df['split']=='train').values
    for cond,feats in X.items():
        clf=train_probe(feats[m_tr],y[m_tr],C=pcfg['C'],max_iter=pcfg['max_iter'])
        for tg,mk in groups_for(df).items():
            p=predict_proba(clf,feats[mk]); au=auroc(y[mk],p); er=eer(y[mk],p)
            log_result(condition=cond,train_gen='stylegan',test_gen=tg,corruption='none',seed=seed,metric='AUROC',value=round(au,4))
            log_result(condition=cond,train_gen='stylegan',test_gen=tg,corruption='none',seed=seed,metric='EER',value=round(er,4))
            rows.append({'seed':seed,'condition':cond,'test_gen':tg,'AUROC':au,'EER':er})
r=pd.DataFrame(rows)
print(r.groupby(['condition','test_gen'])[['AUROC','EER']].agg(['mean','std']).round(4).to_string())

## 4. Bootstrap CI (seed 13, cross-generator)

In [ ]:
df=make_splits(base.copy(), train_generators=['stylegan'], test_unseen=['stable_diffusion'], seed=13)
y=df['label'].values; m_tr=(df['split']=='train').values; mk=groups_for(df)['stable_diffusion']
for cond,feats in X.items():
    clf=train_probe(feats[m_tr],y[m_tr],C=pcfg['C'],max_iter=pcfg['max_iter'])
    p=predict_proba(clf,feats[mk]); pt,lo,hi=bootstrap_ci(y[mk],p,metric_fn=auroc,n=2000)
    print(f'{cond} cross-gen AUROC={pt:.4f} 95% CI [{lo:.4f},{hi:.4f}]')
    log_result(condition=cond,train_gen='stylegan',test_gen='stable_diffusion',corruption='none',seed=13,metric='AUROC_CI95',value=f'[{lo:.4f},{hi:.4f}]')
print('✅ Day 7 gate: mean±spread + bootstrap CIs')